# Target Point

This example demonstrates how to use the eose `TargetPoint` object.

In [ ]:
from eose.targets import TargetPoint
from eose.geometry import Feature, FeatureCollection

import geopandas as gpd
from pydantic import ValidationError

# example constructing an object model
p1 = TargetPoint(position=[0, 0, 0], id=1, crs="EPSG:4326")
display(p1.model_dump_json())  # display EOSE JSON
display(p1.as_feature().model_dump_json())  # display GeoJSON

# example transforming from a GeoJSON object
p2 = TargetPoint.from_feature(
    Feature.model_validate(
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-45, 45, 0]},
            "properties": {"id": 2, "crs": "EPSG:4326"},
        }
    )
)
display(p2.model_dump_json())  # display EOSE JSON
display(p2.as_feature().model_dump_json())  # display GeoJSON

# example validation error
try:
    TargetPoint(position=[-181, 0, 0])
except ValidationError as err:
    display(err)

# example transformation to a GeoDataFrame
targets = gpd.GeoDataFrame.from_features(
    FeatureCollection(features=[p.as_feature() for p in [p1, p2]]),
    crs="EPSG:4326",  # bugfix rquired for geopandas < 0.14
)
display(targets)  # display GeoDataFrame

In [ ]:
from eose.grids import UniformAngularGrid
from shapely.geometry import box, mapping

# constructing an object model
g = UniformAngularGrid(
    delta_latitude=5,
    delta_longitude=5,
    region=mapping(box(-170, 0, -50, 90)),
    crs="EPSG:4326",
)
display(g.model_dump_json())  # display EOSE JSON
display(g.as_features().model_dump_json())  # display GeoJSON

# example transformation to a GeoDataFrame
grid = gpd.GeoDataFrame.from_features(
    g.as_features(), crs="EPSG:4326"  # bugfix rquired for geopandas < 0.14
)
display(grid)  # display GeoDataFrame

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import cartopy.crs as ccrs

# example composite plot using GeoDataFrames
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})
grid.plot(ax=ax, markersize=1, color="b", transform=ccrs.PlateCarree())
targets.plot(ax=ax, markersize=2, color="r", transform=ccrs.PlateCarree())
ax.set_global()
ax.coastlines()
plt.show()